# Retail Investor Flows — Cash Example

This notebook demonstrates how to retrieve and analyse **cash retail flow data** using:

- REST API  
- Official Python SDK (`vanda-api`)  

Covered:

- Authentication (using Login & Password + Bearer Token)
- Series endpoint (single security)  
- Aggregates endpoint  
- Daily, Weekly, Monthly data  
- Intraday data (10min, 30min, 1h)  

# Authentication

## Login Using Email & Password

In [ ]:
import os
import requests
import pandas as pd
from datetime import date
from vanda import VandaClient


VANDA_EMAIL = "VANDA_EMAIL" # <-- Replace with your email
VANDA_PASSWORD = "VANDA_PASSWORD"   # <-- Replace with your password

if VANDA_EMAIL: os.environ["VANDA_EMAIL"] = VANDA_EMAIL

if VANDA_PASSWORD: os.environ["VANDA_PASSWORD"] = VANDA_PASSWORD


API_BASE = "https://api.vanda-analytics.com"
TEST_TOKEN_URL = f"{API_BASE}/series/test/token"

email = VANDA_EMAIL or os.getenv("VANDA_EMAIL")
password = VANDA_PASSWORD or os.getenv("VANDA_PASSWORD")

if not email or not password:
    raise ValueError(
        "Please either:\n"
        "1) Set VANDA_EMAIL and VANDA_PASSWORD in the config cell, OR\n"
        "2) Set them as environment variables."
    )

response = requests.post(
    TEST_TOKEN_URL,
    json={"email": email, "password": password}
)

response.raise_for_status()

TOKEN = response.json()["access_token"]

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json"
}

print("Authentication successful.")

client = VandaClient(
    token=TOKEN,
    base_url=API_BASE
)

Authentication successful.


# REST API

## SERIES (SINGLE SECURITY)

### Helper Functions

In [6]:
def fetch_series_timeseries(
    symbol=None,
    vanda_id=None,
    interval="1d",
    start_date=None,
    end_date=None,
    asset_class="cash",
    fields=None
):
    page = 1
    all_data = []

    while True:
        params = {
            "interval": interval,
            "asset_class": asset_class,
            "records_per_page": 2000,
            "page_number": page,
        }

        if symbol:
            params["symbol"] = symbol
        if vanda_id:
            params["vanda_id"] = vanda_id
        if start_date:
            params["start_date"] = start_date
        if end_date:
            params["end_date"] = end_date
        if fields:
            params["fields"] = fields

        response = requests.get(
            f"{API_BASE}/series/timeseries",
            headers=HEADERS,
            params=params,
        )

        response.raise_for_status()
        data = response.json()

        # Async job triggered
        if isinstance(data, dict) and "job_id" in data:
            print(f"Async job triggered: {data['job_id']}")
            return data

        results = data.get("results", [])
        all_data.extend(results)

        pagination = data.get("pagination", {})
        if not pagination.get("has_next_page"):
            break

        page += 1

    df = pd.DataFrame(all_data)

    if df.empty:
        return df

    if "ts" in df.columns:
        df["ts"] = pd.to_datetime(df["ts"])

    if fields:
        keep_cols = ["ts", "symbol", "vanda_id"] + fields
        existing_cols = [col for col in keep_cols if col in df.columns]
        df = df[existing_cols]

    return df

def clean_intraday_us_equities(df):
    if df.empty:
        return df

    df["ts"] = pd.to_datetime(df["ts"], utc=True)

    # Convert to New York time (handles EST/EDT automatically)
    df["ts_est"] = df["ts"].dt.tz_convert("America/New_York")

    # Keep only regular US market hours (09:30–16:00)
    df = df[
        (df["ts_est"].dt.time >= pd.to_datetime("09:30").time()) &
        (df["ts_est"].dt.time <= pd.to_datetime("16:00").time())
    ]

    flow_cols = [
        col for col in df.columns
        if "turnover" in col
    ]

    if flow_cols:
        df = df[~(df[flow_cols].fillna(0).sum(axis=1) == 0)]

    return df.reset_index(drop=True)

### Daily Example - Retail Flows on Single Securities (Cash)

In [7]:
df_daily = fetch_series_timeseries(
    symbol="AAPL",
    interval="1d",
    start_date="2024-01-01",
    fields=[
        "retail_buy_turnover",
        "retail_sell_turnover",
        "retail_net_turnover",
    ]
)

df_daily.head()

,ts,symbol,vanda_id,retail_buy_turnover,retail_sell_turnover,retail_net_turnover
0,2026-02-23 00:00:00+00:00,AAPL,VNDA1000013,99897690.55,1.077085e+08,-7810847.93
1,2026-02-20 00:00:00+00:00,AAPL,VNDA1000013,35929515.78,4.164951e+07,-5719992.48
2,2026-02-19 00:00:00+00:00,AAPL,VNDA1000013,33035534.55,4.210200e+07,-9066468.37
3,2026-02-18 00:00:00+00:00,AAPL,VNDA1000013,46160757.53,4.288678e+07,3273982.29
4,2026-02-17 00:00:00+00:00,AAPL,VNDA1000013,49098282.02,6.501924e+07,-15920960.32


### Weekly Example - Retail Flows on Single Securities (Cash)

In [ ]:
df_weekly = fetch_series_timeseries(
    symbol="AAPL",
    interval="1w",
    start_date="2024-01-01",
    fields=[
        "retail_buy_turnover",
        "retail_sell_turnover",
        "retail_net_turnover",
    ]
)

df_weekly.head()

,ts,symbol,vanda_id,retail_net_turnover
0,2026-02-27 00:00:00+00:00,AAPL,VNDA1000013,-7.810848e+06
1,2026-02-20 00:00:00+00:00,AAPL,VNDA1000013,-2.743344e+07
2,2026-02-13 00:00:00+00:00,AAPL,VNDA1000013,4.673223e+07
3,2026-02-06 00:00:00+00:00,AAPL,VNDA1000013,-1.474232e+07
4,2026-01-30 00:00:00+00:00,AAPL,VNDA1000013,1.191836e+08


### Monthly Example - Retail Flows on Single Securities (Cash)

In [27]:
df_monthly = fetch_series_timeseries(
    symbol="AAPL",
    interval="1m",
    start_date="2023-01-01",
    fields=[
        "retail_buy_turnover",
        "retail_sell_turnover",
        "retail_net_turnover",
    ]
)

df_monthly.head()

,ts,symbol,vanda_id,retail_buy_turnover,retail_sell_turnover,retail_net_turnover
0,2026-02-28,AAPL,VNDA1000013,9.337313e+08,9.369857e+08,-3.254375e+06
1,2026-01-31,AAPL,VNDA1000013,1.239023e+09,8.692172e+08,3.698054e+08
2,2025-12-31,AAPL,VNDA1000013,1.033338e+09,7.703159e+08,2.630220e+08
3,2025-11-30,AAPL,VNDA1000013,9.902580e+08,8.817514e+08,1.085066e+08
4,2025-10-31,AAPL,VNDA1000013,1.407386e+09,1.075944e+09,3.314416e+08


### Intraday Example (10 Min, 30 Min, 1 Hour) - Retail Flows on Single Securities (Cash)

In [36]:
intraday_intervals = ["10min", "30min", "1h"]
intraday_data = {}

for interval in intraday_intervals:

    df = fetch_series_timeseries(
        symbol="NVDA",
        interval=interval,
        start_date="2026-02-23",
        fields=[
            "retail_buy_turnover",
            "retail_sell_turnover",
            "retail_net_turnover",
        ],
    )

    df = clean_intraday_us_equities(df)

    intraday_data[interval] = df


df_10min = intraday_data["10min"]
df_30min = intraday_data["30min"]
df_1h    = intraday_data["1h"]

df_1h.head()

,ts,symbol,vanda_id,retail_buy_turnover,retail_sell_turnover,retail_net_turnover,ts_est
0,2026-02-23 21:00:00+00:00,NVDA,VNDA1005119,1.173794e+08,1.110381e+08,6341265.98,2026-02-23 16:00:00-05:00
1,2026-02-23 20:00:00+00:00,NVDA,VNDA1005119,9.668911e+07,8.089882e+07,15790295.72,2026-02-23 15:00:00-05:00
2,2026-02-23 19:00:00+00:00,NVDA,VNDA1005119,9.820048e+07,1.020531e+08,-3852607.20,2026-02-23 14:00:00-05:00
3,2026-02-23 18:00:00+00:00,NVDA,VNDA1005119,1.078321e+08,9.442056e+07,13411583.02,2026-02-23 13:00:00-05:00
4,2026-02-23 17:00:00+00:00,NVDA,VNDA1005119,1.421337e+08,1.350522e+08,7081555.18,2026-02-23 12:00:00-05:00


## SERIES (AGGREGATES)

### Helper Functions

In [93]:
def fetch_aggregates_timeseries(
    aggregate_ids,
    interval="1d",
    start_date=None,
    end_date=None,
    asset_class="cash",
    fields=None,
    page_size=2000,
):
    page = 1
    all_data = []

    while True:
        params = {
            "aggregate_ids": aggregate_ids,
            "interval": interval,
            "asset_class": asset_class,
            "records_per_page": page_size,
            "page_number": page,
            "order": "desc",
        }

        if start_date:
            params["start_date"] = start_date
        if end_date:
            params["end_date"] = end_date
        if fields:
            params["fields"] = fields

        response = requests.get(
            "https://api.vanda-analytics.com/aggregates/timeseries",
            headers=HEADERS,
            params=params,
        )

        response.raise_for_status()
        data = response.json()

        results = data.get("data", [])
        all_data.extend(results)

        if "pagination" not in data:
            break

        if not data["pagination"].get("has_next_page"):
            break

        page += 1

    df = pd.DataFrame(all_data)

    if not df.empty and "ts" in df.columns:
        df["ts"] = pd.to_datetime(df["ts"], utc=True)

    return df

def clean_aggregate_df(df, fields=None, drop_metadata=True, set_index=True):

    if df.empty:
        return df

    # Ensure datetime
    if "ts" in df.columns:
        df["ts"] = pd.to_datetime(df["ts"], utc=True)

    # Convert numeric flow fields
    flow_cols = [
        "retail_buy_turnover",
        "retail_sell_turnover",
        "retail_net_turnover",
        "retail_total_turnover",
    ]

    for col in flow_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep only selected fields
    if fields:
        keep_cols = ["ts", "id", "type", "subtype"] + fields
        keep_cols = [c for c in keep_cols if c in df.columns]
        df = df[keep_cols]

    # Drop noisy metadata columns
    if drop_metadata:
        drop_cols = [
            "px",
            "px_chg",
            "market_hours",
            "is_active",
            "updated_time",
            "asset_class",
        ]
        df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    df = df.sort_values("ts")

    if set_index and "ts" in df.columns:
        df = df.set_index("ts")

    return df

def clean_intraday_us_equities(df):
    if df.empty:
        return df

    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    df["ts_est"] = df["ts"].dt.tz_convert("America/New_York")

    df = df[
        (df["ts_est"].dt.time >= pd.to_datetime("09:30").time()) &
        (df["ts_est"].dt.time <= pd.to_datetime("16:00").time())
    ]

    flow_cols = [
        col for col in df.columns
        if "turnover" in col
    ]

    if flow_cols:
        df = df[~(df[flow_cols].fillna(0).sum(axis=1) == 0)]

    return df.sort_values("ts_est").reset_index(drop=True)

def fetch_multiple_aggregates_timeseries(
    aggregate_ids,
    interval="1d",
    start_date=None,
    end_date=None,
    asset_class="cash",
    fields=None,
    page_size=2000,
):
    # Ensure comma-separated string
    if isinstance(aggregate_ids, list):
        aggregate_ids = ",".join(str(x) for x in aggregate_ids)

    page = 1
    all_data = []

    while True:
        params = {
            "aggregate_ids": aggregate_ids,
            "interval": interval,
            "asset_class": asset_class,
            "records_per_page": page_size,
            "page_number": page,
            "order": "desc",
        }

        if start_date:
            params["start_date"] = start_date
        if end_date:
            params["end_date"] = end_date
        if fields:
            params["fields"] = fields

        response = requests.get(
            "https://api.vanda-analytics.com/aggregates/timeseries",
            headers=HEADERS,
            params=params,
        )

        response.raise_for_status()
        data = response.json()

        results = data.get("data", [])
        all_data.extend(results)

        if "pagination" not in data:
            break

        if not data["pagination"].get("has_next_page"):
            break

        page += 1

    df = pd.DataFrame(all_data)

    if not df.empty:
        df["ts"] = pd.to_datetime(df["ts"], utc=True)
        df = df.sort_values("ts")

        # Convert numeric columns
        numeric_cols = [
            "retail_buy_turnover",
            "retail_sell_turnover",
            "retail_net_turnover",
            "retail_total_turnover",
        ]

        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.reset_index(drop=True)

### Daily Example - Retail Flows on Aggregates (Cash)

In [96]:
df_agg_daily = fetch_aggregates_timeseries(
    aggregate_ids=2000003,
    interval="1d",
    start_date="2024-01-01",
    end_date="2024-12-31",
    fields=["retail_net_turnover"]
)

df_agg_daily = clean_aggregate_df(df_agg_daily)

df_agg_daily.head()

,id,type,subtype,retail_buy_turnover,retail_sell_turnover,retail_net_turnover,retail_total_turnover
ts,,,,,,,
2024-01-02 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),4.470870e+08,4.113894e+08,35697633.01,8.584764e+08
2024-01-03 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),4.928993e+08,4.712747e+08,21624687.91,9.641740e+08
2024-01-04 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),4.422859e+08,3.863400e+08,55945867.15,8.286259e+08
2024-01-05 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),3.934697e+08,3.710651e+08,22404647.75,7.645348e+08
2024-01-08 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),4.204202e+08,3.813173e+08,39102859.72,8.017375e+08


### Weekly Example - Retail Flows on Aggregates (Cash)

In [97]:
df_agg_weekly = fetch_aggregates_timeseries(
    aggregate_ids=2000003,
    interval="1w",
    start_date="2023-01-01",
    fields=["retail_net_turnover"]
)

df_agg_weekly = clean_aggregate_df(df_agg_weekly)
df_agg_weekly.head()

,id,type,subtype,retail_buy_turnover,retail_sell_turnover,retail_net_turnover,retail_total_turnover
ts,,,,,,,
2023-01-06 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),1.269994e+09,1.191649e+09,7.834525e+07,2.461643e+09
2023-01-13 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),1.194952e+09,1.108188e+09,8.676400e+07,2.303140e+09
2023-01-27 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),1.270855e+09,1.243506e+09,2.734928e+07,2.514361e+09
2023-02-03 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),1.777815e+09,1.697086e+09,8.072962e+07,3.474901e+09
2023-02-10 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),2.017684e+09,1.890115e+09,1.275699e+08,3.907799e+09


### Monthly Example - Retail Flows on Aggregates (Cash)

In [98]:
df_agg_monthly = fetch_aggregates_timeseries(
    aggregate_ids=2000003,
    interval="1m",
    start_date="2022-01-01",
    fields=["retail_net_turnover"]
)

df_agg_monthly = clean_aggregate_df(df_agg_monthly)
df_agg_monthly.head()

,id,type,subtype,retail_buy_turnover,retail_sell_turnover,retail_net_turnover,retail_total_turnover
ts,,,,,,,
2022-01-31 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),6.531040e+09,5.580800e+09,9.502393e+08,1.211184e+10
2022-02-28 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),1.006931e+10,8.682451e+09,1.386857e+09,1.875176e+10
2022-03-31 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),9.119267e+09,7.785994e+09,1.333273e+09,1.690526e+10
2022-04-30 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),7.226553e+09,6.294218e+09,9.323357e+08,1.352077e+10
2022-05-31 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),5.444774e+09,4.924815e+09,5.199584e+08,1.036959e+10


### Intraday Example (10min, 30min, 1H) - Retail Flows on Aggregates (Cash)

In [99]:
df_10min = fetch_aggregates_timeseries(
    aggregate_ids=2000003,
    interval="10min",
    start_date="2026-02-23",
    fields=["retail_net_turnover"]
)

df_10min = clean_intraday_us_equities(df_10min)

df_30min = fetch_aggregates_timeseries(
    aggregate_ids=2000003,
    interval="30min",
    start_date="2026-02-23",
    fields=["retail_net_turnover"]
)

df_30min = clean_intraday_us_equities(df_30min)

df_1h = fetch_aggregates_timeseries(
    aggregate_ids=2000003,
    interval="1h",
    start_date="2026-02-23",
    fields=["retail_net_turnover"]
)

df_1h = clean_intraday_us_equities(df_1h)
df_1h.head()

,ts,id,type,subtype,market_hours,is_active,px,px_chg,updated_time,asset_class,retail_buy_turnover,retail_sell_turnover,retail_net_turnover,retail_total_turnover,ts_est
0,2026-02-23 15:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),regular_market,True,51.6993,-0.0104,2026-02-24T06:11:32.167396Z,cash,62651343.23,61648367.56,1002975.77,124299710.87,2026-02-23 10:00:00-05:00
1,2026-02-23 16:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),regular_market,True,50.8819,-0.0158,2026-02-24T06:11:32.167396Z,cash,140107920.05,119854206.84,20253713.31,259962126.88,2026-02-23 11:00:00-05:00
2,2026-02-23 17:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),regular_market,True,50.8600,-0.0004,2026-02-24T06:11:32.167396Z,cash,127321709.30,109950322.55,17371387.07,237272031.55,2026-02-23 12:00:00-05:00
3,2026-02-23 18:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),regular_market,True,50.7840,-0.0015,2026-02-24T06:11:32.167396Z,cash,99060955.53,90097184.52,8963771.31,189158140.19,2026-02-23 13:00:00-05:00
4,2026-02-23 19:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),regular_market,True,50.7937,0.0002,2026-02-24T06:11:32.167396Z,cash,90813797.44,83924698.75,6889098.81,174738496.09,2026-02-23 14:00:00-05:00


### Multi-Aggregates 

In [95]:
fields = [
    "retail_buy_turnover",
    "retail_sell_turnover",
    "retail_net_turnover",
]

df_multi_agg = fetch_multiple_aggregates_timeseries(
    aggregate_ids=[2000003, 2000004, 2000005],
    interval="1d",
    start_date="2024-01-01",
    end_date="2024-12-31",
    fields=fields,
)

df_multi_agg = clean_aggregate_df(
    df_multi_agg,
    fields=fields
)

df_multi_agg.head()

,id,type,subtype,retail_buy_turnover,retail_sell_turnover,retail_net_turnover
ts,,,,,,
2024-01-02 00:00:00+00:00,2000003,Benchmark Sector,Russell-1000 (Financials),4.470870e+08,4.113894e+08,35697633.01
2024-01-02 00:00:00+00:00,2000005,Benchmark Sector,Russell-3000 (Financials),4.919958e+08,4.552160e+08,36779774.27
2024-01-02 00:00:00+00:00,2000004,Benchmark Sector,Russell-2000 (Financials),4.490880e+07,4.382666e+07,1082141.26
2024-01-03 00:00:00+00:00,2000005,Benchmark Sector,Russell-3000 (Financials),5.337261e+08,5.088794e+08,24846672.93
2024-01-03 00:00:00+00:00,2000004,Benchmark Sector,Russell-2000 (Financials),4.082671e+07,3.760473e+07,3221985.02


# Python SDK

## Single Security - Synchronous

### Helper Functions

In [53]:
def clean_timeseries_df(df, fields=None):
    if isinstance(df, list):
        df = pd.DataFrame(df)

    if df.empty:
        return df

    # Timestamp
    if "ts" in df.columns:
        df["ts"] = pd.to_datetime(df["ts"], utc=True)
        df = df.sort_values("ts")

    # Keep only relevant columns
    base_cols = ["ts", "symbol", "vanda_id"]

    if fields:
        keep_cols = base_cols + fields
        keep_cols = [col for col in keep_cols if col in df.columns]
        df = df[keep_cols]

    # Convert numeric columns
    numeric_cols = df.select_dtypes(include="object").columns

    for col in numeric_cols:
        if col not in ["ts", "symbol", "vanda_id"]:
            df[col] = pd.to_numeric(df[col], errors="ignore")

    return df.reset_index(drop=True)


def clean_intraday_us_equities(df):
    if df.empty:
        return df

    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    df["ts_est"] = df["ts"].dt.tz_convert("America/New_York")

    df = df[
        (df["ts_est"].dt.time >= pd.to_datetime("09:30").time()) &
        (df["ts_est"].dt.time <= pd.to_datetime("16:00").time())
    ]

    flow_cols = [
        col for col in df.columns
        if "turnover" in col
    ]

    if flow_cols:
        df = df[~(df[flow_cols].fillna(0).sum(axis=1) == 0)]

    return df.sort_values("ts_est").reset_index(drop=True)

### Daily Example - Retail Flows on Single Security (Cash)

In [48]:
fields = [
    "retail_buy_turnover",
    "retail_sell_turnover",
    "retail_net_turnover",
]

df_daily = client.get_timeseries(
    symbol="AAPL",
    interval="1d",
    start_date="2024-01-01",
    end_date="2024-12-31",
    fields=fields,
    asset_class="cash",
)

df_daily = clean_timeseries_df(df_daily, fields=fields)

df_daily.head()


,ts,symbol,vanda_id,retail_buy_turnover,retail_sell_turnover,retail_net_turnover
0,2024-01-02 00:00:00+00:00,AAPL,VNDA1000013,1.590939e+08,9.883708e+07,60256806.60
1,2024-01-03 00:00:00+00:00,AAPL,VNDA1000013,1.276029e+08,1.209753e+08,6627587.61
2,2024-01-04 00:00:00+00:00,AAPL,VNDA1000013,2.037890e+08,1.130444e+08,90744656.52
3,2024-01-05 00:00:00+00:00,AAPL,VNDA1000013,1.734709e+08,1.168529e+08,56618093.47
4,2024-01-08 00:00:00+00:00,AAPL,VNDA1000013,1.323559e+08,9.546825e+07,36887641.19


### Weekly Example - Retail Flows on Single Security (Cash)

In [50]:
fields = [
    "retail_buy_turnover",
    "retail_sell_turnover",
    "retail_net_turnover",
]

df_weekly = client.get_timeseries(
    symbol="AAPL",
    interval="1w",
    start_date="2023-01-01",
    end_date="2024-12-31",
    fields=fields,
    asset_class="cash",
)

df_weekly = clean_timeseries_df(df_weekly, fields=fields)

df_weekly.head()

,ts,symbol,vanda_id,retail_buy_turnover,retail_sell_turnover,retail_net_turnover
0,2023-01-06 00:00:00+00:00,AAPL,VNDA1000013,6.824060e+08,3.791613e+08,3.032448e+08
1,2023-01-13 00:00:00+00:00,AAPL,VNDA1000013,7.866741e+08,4.526431e+08,3.340310e+08
2,2023-01-20 00:00:00+00:00,AAPL,VNDA1000013,7.209971e+08,5.303407e+08,1.906564e+08
3,2023-01-27 00:00:00+00:00,AAPL,VNDA1000013,1.036063e+09,8.138137e+08,2.222489e+08
4,2023-02-03 00:00:00+00:00,AAPL,VNDA1000013,1.198118e+09,9.956005e+08,2.025172e+08


### Monthly Example - Retail Flows on Single Security (Cash)

In [52]:
fields = [
    "retail_buy_turnover",
    "retail_sell_turnover",
    "retail_net_turnover",
]

df_monthly = client.get_timeseries(
    symbol="AAPL",
    interval="1m",
    start_date="2020-01-01",
    end_date="2024-12-31",
    fields=fields,
    asset_class="cash",
)

df_monthly = clean_timeseries_df(df_monthly, fields=fields)

df_monthly.head()

,ts,symbol,vanda_id,retail_buy_turnover,retail_sell_turnover,retail_net_turnover
0,2020-01-31 00:00:00+00:00,AAPL,VNDA1000013,8.199557e+09,7.929533e+09,2.700242e+08
1,2020-02-29 00:00:00+00:00,AAPL,VNDA1000013,5.536925e+09,5.312328e+09,2.245969e+08
2,2020-03-31 00:00:00+00:00,AAPL,VNDA1000013,1.092875e+10,1.030666e+10,6.220951e+08
3,2020-04-30 00:00:00+00:00,AAPL,VNDA1000013,5.752433e+09,5.249030e+09,5.034027e+08
4,2020-05-31 00:00:00+00:00,AAPL,VNDA1000013,5.207144e+09,4.871916e+09,3.352276e+08


### Intraday Example (10min, 30min, 1H) - Retail Flows on Single Security (Cash)

In [55]:
fields = [
    "retail_buy_turnover",
    "retail_sell_turnover",
    "retail_net_turnover",
]

df_10min = client.get_timeseries(
    symbol="AAPL",
    interval="10min",
    start_date="2026-02-16",
    end_date="2026-02-18",
    fields=fields,
    asset_class="cash",
)

df_10min = clean_timeseries_df(df_10min, fields=fields)
df_10min = clean_intraday_us_equities(df_10min)

df_30min = client.get_timeseries(
    symbol="AAPL",
    interval="30min",
    start_date="2026-02-16",
    end_date="2026-02-18",
    fields=fields,
    asset_class="cash",
)

df_30min = clean_timeseries_df(df_30min, fields=fields)
df_30min = clean_intraday_us_equities(df_30min)

df_1h = client.get_timeseries(
    symbol="AAPL",
    interval="1h",
    start_date="2026-02-16",
    end_date="2026-02-18",
    fields=fields,
    asset_class="cash",
)

df_1h = clean_timeseries_df(df_1h, fields=fields)
df_1h = clean_intraday_us_equities(df_1h)

df_1h.head()

,ts,symbol,vanda_id,retail_buy_turnover,retail_sell_turnover,retail_net_turnover,ts_est
0,2026-02-17 15:00:00+00:00,AAPL,VNDA1000013,3023753.91,3256958.46,-233204.55,2026-02-17 10:00:00-05:00
1,2026-02-17 16:00:00+00:00,AAPL,VNDA1000013,2423379.49,6954084.03,-4530704.54,2026-02-17 11:00:00-05:00
2,2026-02-17 17:00:00+00:00,AAPL,VNDA1000013,4279395.85,10172487.52,-5893091.68,2026-02-17 12:00:00-05:00
3,2026-02-17 18:00:00+00:00,AAPL,VNDA1000013,4350221.50,6522750.41,-2172528.90,2026-02-17 13:00:00-05:00
4,2026-02-17 19:00:00+00:00,AAPL,VNDA1000013,10690156.01,11928734.61,-1238578.60,2026-02-17 14:00:00-05:00


## Single Security - Asynchronous

### Helper Functions

In [ ]:
def get_timeseries_many_safe(**kwargs):
    result = client.get_timeseries_many(**kwargs)

    if isinstance(result, dict) and "job_id" in result:
        print("Async job triggered. Waiting for completion...")
        return client.wait_for_job(result["job_id"])

    return result


def clean_timeseries_df(df, fields=None):
    if isinstance(df, list):
        df = pd.DataFrame(df)

    if df.empty:
        return df

    if "ts" in df.columns:
        df["ts"] = pd.to_datetime(df["ts"], utc=True)
        df = df.sort_values("ts")

    base_cols = ["ts", "symbol", "vanda_id"]

    if fields:
        keep_cols = base_cols + fields
        keep_cols = [col for col in keep_cols if col in df.columns]
        df = df[keep_cols]

    numeric_cols = df.select_dtypes(include="object").columns

    for col in numeric_cols:
        if col not in ["ts", "symbol", "vanda_id"]:
            df[col] = pd.to_numeric(df[col], errors="ignore")

    return df.reset_index(drop=True)

def clean_intraday_us_equities(df):
    if df.empty:
        return df

    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    df["ts_est"] = df["ts"].dt.tz_convert("America/New_York")

    df = df[
        (df["ts_est"].dt.time >= pd.to_datetime("09:30").time()) &
        (df["ts_est"].dt.time <= pd.to_datetime("16:00").time())
    ]

    flow_cols = [
        col for col in df.columns
        if "turnover" in col
    ]

    if flow_cols:
        df = df[~(df[flow_cols].fillna(0).sum(axis=1) == 0)]

    return df.sort_values("ts_est").reset_index(drop=True)

### Multi-Security Request

In [68]:
fields = [
    "retail_buy_turnover",
    "retail_sell_turnover",
    "retail_net_turnover",
]

data = get_timeseries_many_safe(
    symbols=["AAPL", "MSFT", "NVDA"],
    interval="1d",
    start_date="2020-01-01",
    end_date="2024-12-31",
    fields=fields,
    asset_class="cash",
)

df_portfolio = clean_timeseries_df(data, fields=fields)

df_portfolio.head()

Returned synchronously.


,ts,symbol,vanda_id,retail_buy_turnover,retail_sell_turnover,retail_net_turnover
0,2020-01-02 00:00:00+00:00,AAPL,VNDA1000013,4.052884e+08,3.815159e+08,23772461.55
1,2020-01-02 00:00:00+00:00,MSFT,VNDA1004668,9.013124e+07,8.431129e+07,5819948.81
2,2020-01-02 00:00:00+00:00,NVDA,VNDA1005119,4.370413e+07,4.237179e+07,1332338.84
3,2020-01-03 00:00:00+00:00,AAPL,VNDA1000013,4.523610e+08,4.453216e+08,7039479.56
4,2020-01-03 00:00:00+00:00,MSFT,VNDA1004668,6.480123e+07,5.984856e+07,4952672.42


### Multi-Year Historical Data

In [ ]:
fields = ["retail_buy_turnover", "retail_sell_turnover", "retail_net_turnover"]

data = get_timeseries_many_safe(
    symbols=["AAPL"],
    interval="1d",
    start_date="2012-01-01",
    end_date="2024-12-31",
    fields=fields,
    asset_class="cash",
)

df_backfill = clean_timeseries_df(data, fields=fields)

df_backfill.head()

Returned synchronously.


,ts,symbol,vanda_id,retail_buy_turnover,retail_sell_turnover,retail_net_turnover
0,2012-01-03 00:00:00+00:00,AAPL,VNDA1000013,1.347717e+08,1.492730e+08,-14501392.60
1,2012-01-04 00:00:00+00:00,AAPL,VNDA1000013,1.402527e+08,1.222826e+08,17970063.62
2,2012-01-05 00:00:00+00:00,AAPL,VNDA1000013,1.553080e+08,1.316138e+08,23694123.79
3,2012-01-06 00:00:00+00:00,AAPL,VNDA1000013,1.669381e+08,1.630877e+08,3850316.49
4,2012-01-09 00:00:00+00:00,AAPL,VNDA1000013,2.307922e+08,2.292830e+08,1509263.05


### Multi-Security High Frequency Intraday

In [80]:
fields = ["retail_buy_turnover", "retail_sell_turnover", "retail_net_turnover"]

data = get_timeseries_many_safe(
    symbols=["AAPL", "NVDA", "TSLA", "META"],
    interval="10min",
    start_date="2026-02-01",
    end_date="2026-02-15",
    fields=fields,
    asset_class="cash",
)

df_intraday = clean_timeseries_df(data, fields=fields)
df_intraday = clean_intraday_us_equities(df_intraday)

df_intraday.head()

,ts,symbol,vanda_id,retail_buy_turnover,retail_sell_turnover,retail_net_turnover,ts_est
0,2026-02-02 14:40:00+00:00,AAPL,VNDA1000013,2079392.32,2256250.55,-176858.23,2026-02-02 09:40:00-05:00
1,2026-02-02 14:40:00+00:00,TSLA,VNDA1010382,10588627.19,8757085.37,1831541.82,2026-02-02 09:40:00-05:00
2,2026-02-02 14:40:00+00:00,META,VNDA1011687,1287136.29,1449008.84,-161872.54,2026-02-02 09:40:00-05:00
3,2026-02-02 14:40:00+00:00,NVDA,VNDA1005119,3498810.28,3403234.85,95575.43,2026-02-02 09:40:00-05:00
4,2026-02-02 14:50:00+00:00,AAPL,VNDA1000013,1425477.69,1554710.56,-129232.87,2026-02-02 09:50:00-05:00


# Documentation

Full documentation is available at:

- REST API: https://docs.vanda-analytics.com/
- Python SDK (`vanda-api`): https://pypi.org/project/vanda-api/